### Este notebook va a ser para hacer la exploración de DATD, preprocesamiento, División
Después se hará la baseline en ingles con los trains y Berta y Roberta, tokenización, padding, balanceo de clases, semilla 

# Cargar DATD

In [ ]:
# Hay tres csvs para DATD, los voy a unir para Crear uno solo llamado DATD_total

import pandas as pd

df_ = pd.read_csv('./Datasets RAW/1 TheDATD/DATD_and_DATD+Rand_test.csv')
df__ = pd.read_csv('./Datasets RAW/1 TheDATD/DATD_training.csv')
df___ = pd.read_csv('./Datasets RAW/1 TheDATD/DATD+Rand_training.csv')

df = pd.concat([df_, df__, df___], ignore_index=True)

df.to_csv("./Bases de Datos Completas/1 TheDATD/DATD_total.csv", index=False)

In [ ]:
df = pd.read_csv('./Bases de Datos Completas/1 TheDATD/DATD_total.csv')

In [ ]:
df

# Exploración y limpieza de datos

In [ ]:
# Información general del DataFrame
print('\n--- Información del DataFrame (nulls, tipos, etc.) ---')
print(df.info())

In [ ]:
#Eliminación de columnas
#df = df.drop(columns=['Unnamed: 0.1','Unnamed: 0','title','author','num_comments','url','selftext_clean']).copy()

# Renombrar columnas
df.columns = ['Text', 'Label']

# Añadir columna con la fuente de la base de datos
df['Source'] = 'DATD'

# Volver a ver información general del DataFrame
print('\n--- Información del DataFrame (nulls, tipos, etc.) ---')
print(df.info())

# Conteo de etiquetas
print('\n--- Conteo de etiquetas únicas ---')
print(df['Label'].unique())

# Renombramos las etiquetas: MENTAL_HEALTH como 1 y OTHER como 0
print('\n--- Renombramos las etiquetas: MENTAL_HEALTH como 1 y OTHER como 0 ---')
df['Label'] = df['Label'].map({'MENTAL_HEALTH': 1, 'OTHER': 0})

# Conteo de instancias por etiqueta
print('\n--- Conteo de instancias según etiquetas ---')
print(df['Label'].value_counts())

# Conteo de valores únicos en la columna Text
print('\n--- Número de variables únicas ---')
print(df['Text'].nunique())

# Mostrar textos duplicados
print('\n--- Variables duplicadas (si existen) y su eliminación  ---')
duplicados = df[df.duplicated(subset='Text', keep=False)]
print(duplicados)

# Eliminar duplicados en la columna Text
df = df.drop_duplicates(subset='Text').copy()

# Verificar nueva información del DataFrame tras eliminar duplicados
print('\n--- Información del DataFrame tras eliminar duplicados ---')
print(df.info())

# Mostrar los primeros registros del DataFrame
print('\n--- Primeros registros del DataFrame ---')
print(df.head())
df

# Preprocesamiento de DATD

In [ ]:
!pip install emoji
import re
import emoji

# Función para sustituir menciones tipo @user
def limpiar_menciones(texto):
    texto = re.sub(r'@\s+', '@', texto)      # corregir menciones con espacio
    texto = re.sub(r'@[^\s]+', '', texto)
    return texto

# Función para sustituir URLs
def limpiar_urls(texto):
    texto = re.sub(r'http\S+|www\S+', '', texto)
    return texto

# Función para eliminar hashtags completos
def limpiar_hashtags(texto):
    texto = re.sub(r'#[^\s]+', '', texto)
    return texto

# Eliminación de Emojis
def limpiar_emojis(texto):
    return emoji.replace_emoji(texto, replace='')

# Función para eliminación de saltos de línea
def eliminar_salto(texto):
    return re.sub('\n','',texto)



# Definir todo el proceso de preprocesamiento básico para Transformers
def preprocesamiento(texto):
    texto = str(texto)

    texto = limpiar_menciones(texto)
    texto = limpiar_urls(texto)
    texto = limpiar_hashtags(texto)
    texto = limpiar_emojis(texto)
    texto = eliminar_salto(texto)
    texto = texto.lower() # Conversión a minúsculas

    return texto

# Aplicar preprocesamiento
df['Text'] = df['Text'].astype(str).apply(preprocesamiento)

# Eliminar instancias vacías después del preprocesamiento
df = df.dropna(subset=['Text'])
df = df[df['Text'].str.strip() != ''].copy()

# Mostrar primeros registros
print('\n--- Primeros registros tras preprocesamiento del texto ---')
print(df.head())

df.to_csv("./Bases de Datos Limpias/1 TheDATD/DATD_limpio.csv", index=False, encoding='utf-8-sig')